# Spatiotemporal Forecasting with Graph-Enformer (GEnformer)

This notebook demonstrates the usage of the `GEnformer` for spatiotemporal data (data with both temporal sequences and spatial topology).

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries
from genformer.models import GEnformer

plt.style.use('seaborn-v0_8-darkgrid')


In [ ]:
# Dummy data generation: 3 spatial nodes
time_steps = 150
num_nodes = 3
data = np.random.randn(time_steps, num_nodes).astype(np.float32)
# Add some spatial and temporal correlation
data[:, 1] += 0.5 * data[:, 0]
data[:, 2] -= 0.3 * data[:, 1]
for i in range(1, time_steps):
    data[i] += 0.2 * data[i-1]

df = pd.DataFrame(data, columns=['Node_0', 'Node_1', 'Node_2'])
series = TimeSeries.from_dataframe(df)

train, val = series[:-30], series[-30:]

plt.figure(figsize=(10, 4))
train.plot()
plt.title('Spatiotemporal Dummy Data (3 Nodes)')
plt.show()


In [ ]:
# Create dummy adjacency matrix (edges)
edges = torch.tensor([
    [0, 1, 1],
    [1, 0, 1],
    [1, 1, 0]
], dtype=torch.float32)

model = GEnformer(
    input_chunk_length=20,
    output_chunk_length=10,
    edges=edges,
    num_nodes=num_nodes,
    num_samples_engression=5,
    n_epochs=2, # Demo
    batch_size=8,
    d_model=64,
    nhead=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    dim_feedforward=128,
    dropout=0.1
)

model.fit(train)


In [ ]:
from genformer.utils import generate_forecasts

# Generate forecasts using the custom spatial method
# Output shape will be (M, T_out, N, D_gcn) where D_gcn is the latent spatial dimension
# The model expects exactly input_chunk_length as the history window
predictions_tensor = generate_forecasts(
    model=model,
    history=train[-20:], # 20 is the input_chunk_length
    m_samples=30,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# We average over the latent spatial dimension to get the 3 node forecasts
predictions_nodes = predictions_tensor.mean(dim=-1).cpu().numpy() # (M, T_out, N)
